In [ ]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
from src.geometric_utils import *
from src.feature_building_utils import *
from sklearn.preprocessing          import OrdinalEncoder
from sklearn.impute                 import IterativeImputer
import toml

In [ ]:
df = pd.read_parquet('data/processed_data/S3-approx-coordinates.parquet')

In [ ]:
docs = toml.load("documentation/feature_docs.toml")

In [ ]:
G = ox.graph.graph_from_bbox(
    bbox=BBOX,
    network_type="all",
    simplify=False
)
nodes, edges = ox.graph_to_gdfs(G)

In [ ]:
feature_name = "num_streets_20"

df[feature_name] = df.apply(
    lambda row: get_nearest_rows(
        nodes[nodes['highway'] == 'crossing'],
        Point(row.x, row.y),
        radius_meters=20
    )['street_count'].sum(),
    axis=1
).astype(int)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Number of nearby streets within 20 meters"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = "Depends on the subject"
docs[feature_name]["created_on"] = "osmnx.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "num_streets_50"

df[feature_name] = df.apply(
    lambda row: get_nearest_rows(
        nodes[nodes['highway'] == 'crossing'],
        Point(row.x, row.y),
        radius_meters=50
    )['street_count'].sum(),
    axis=1
).astype(int)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Number of nearby streets within 50 meters"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = "Depends on the subject"
docs[feature_name]["created_on"] = "osmnx.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "num_traffic_light_200"

df[feature_name] = df.apply(
    lambda row: get_nearest_rows(
        nodes[nodes['highway'] == 'traffic_signals'],
        Point(row.x, row.y),
        radius_meters=200
    ).shape[0],
    axis=1
).astype(int)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Number of nearby traffic lights within 200 meters"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = "Depends on the subject"
docs[feature_name]["created_on"] = "osmnx.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "num_traffic_light_400"

df[feature_name] = df.apply(
    lambda row: get_nearest_rows(
        nodes[nodes['highway'] == 'traffic_signals'],
        Point(row.x, row.y),
        radius_meters=400
    ).shape[0],
    axis=1
).astype(int)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Number of nearby traffic lights within 400 meters"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = "Depends on the subject"
docs[feature_name]["created_on"] = "osmnx.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
edges['lanes'] = edges['lanes'].fillna(0)
edges['oneway'] = edges['oneway'].astype(int)
edges['reversed'] = edges['reversed'].astype(int)
edges['maxspeed'] = edges['maxspeed'].astype(float)

edges.loc[edges["oneway"] == 1, "lanes"] = 1
edges.loc[edges['lanes'] == 0, 'lanes'] = np.nan

In [ ]:
# 1) Define your columns
non_input_cols   = ["osmid", "name", "ref", "geometry", "width", "bridge", "tunnel", "junction"]
categorical_cols = ["highway", "access"]
bool_cols        = ["oneway", "reversed"]
numerical_cols   = ["length", "lanes", "maxspeed"]

input_cols = categorical_cols + bool_cols + numerical_cols

# 2) Make your DataFrame slices
edges_copy        = edges.copy()
X         = edges_copy[input_cols].copy()
mask_foot = X["highway"].isin(["footway", "pedestrian", "unclassified", "steps", "corridor", "path", ])

# 3) Encode all cats & bools
enc = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
X[categorical_cols + bool_cols] = enc.fit_transform(
    X[categorical_cols + bool_cols]
)

# 4) Impute everything (one-shot MICE)
imp = IterativeImputer(
    max_iter=100,
    random_state=0
    # if sklearn ≥1.4 you could add min_value=0 here.
)
X_imputed = pd.DataFrame(
    imp.fit_transform(X),
    columns=input_cols,
    index=X.index
)

# 5) Clamp all numerics ≥ 0
for col in numerical_cols:
    X_imputed[col] = X_imputed[col].clip(lower=0)

# 6) Restore footway rows for lanes & maxspeed to NaN
X_imputed.loc[mask_foot, ["lanes","maxspeed"]] = np.nan

# 7) Decode cats & bools back to original labels
codes = X_imputed[categorical_cols + bool_cols].round().astype(int)
X_imputed[categorical_cols + bool_cols] = enc.inverse_transform(codes)

# 8) Re-attach your untouched columns
result = pd.concat([edges_copy[non_input_cols], X_imputed], axis=1)

In [ ]:
cars = result[~mask_foot]

In [ ]:
feature_name = "average_len_nearby_streets_50"

df[feature_name] = df.apply(
    lambda row: get_nearest_rows(
        edges,
        Point(row.x, row.y),
        radius_meters=50
    )['length'].mean(),
    axis=1
).astype(float)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Average length of nearby streets within 50 meters"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = "Depends on the subject"
docs[feature_name]["created_on"] = "osmnx.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "average_nearby_maxspeed_50"

df[feature_name] = df.apply(
    lambda row: get_nearest_rows(
        cars,
        Point(row.x, row.y),
        radius_meters=50
    )['maxspeed'].mean(),
    axis=1
).astype(float)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Average maxspeed of nearby streets within 50 meters"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = "Depends on the subject"
docs[feature_name]["created_on"] = "osmnx.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "average_nearby_num_lanes_50"

df[feature_name] = df.apply(
    lambda row: get_nearest_rows(
        cars,
        Point(row.x, row.y),
        radius_meters=50
    )['lanes'].mean(),
    axis=1
).astype(float)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Average number of lanes of nearby streets within 50 meters"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = "Depends on the subject"
docs[feature_name]["created_on"] = "osmnx.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "len_nearby_residential_street_50"

df[feature_name] = df.apply(
    lambda row: get_nearest_rows(
        edges[edges['highway'] == 'residential'],
        Point(row.x, row.y),
        50
    )['length'].sum(),
    axis=1
).astype(float)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Length of nearby residential streets within 50 meters"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = "Depends on the subject"
docs[feature_name]["created_on"] = "osmnx.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "len_nearby_residential_street_150"

df[feature_name] = df.apply(
    lambda row: get_nearest_rows(
        edges[edges['highway'] == 'residential'],
        Point(row.x, row.y),
        150
    )['length'].sum(),
    axis=1
).astype(float)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Length of nearby residential streets within 150 meters"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = "Depends on the subject"
docs[feature_name]["created_on"] = "osmnx.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "len_nearby_pedestrian_street_15"

df[feature_name] = df.apply(
    lambda row: get_nearest_rows(
        edges[edges['highway'] == 'pedestrian'],
        Point(row.x, row.y),
        15
    )['length'].sum(),
    axis=1
).astype(float)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Length of nearby pedestrian streets within 15 meters"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = "Depends on the subject"
docs[feature_name]["created_on"] = "osmnx.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "len_nearby_pedestrian_street_50"

df[feature_name] = df.apply(
    lambda row: get_nearest_rows(
        edges[mask_foot],
        Point(row.x, row.y),
        50
    )['length'].sum(),
    axis=1
).astype(float)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Length of nearby pedestrian streets within 50 meters"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = "Depends on the subject"
docs[feature_name]["created_on"] = "osmnx.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "len_nearby_service_street_50"

df[feature_name] = df.apply(
    lambda row: get_nearest_rows(
        edges[edges['highway'] == 'service'],
        Point(row.x, row.y),
        50
    )['length'].sum(),
    axis=1
).astype(float)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Length of nearby service streets within 50 meters"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = "Depends on the subject"
docs[feature_name]["created_on"] = "osmnx.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "len_nearby_service_street_150"

df[feature_name] = df.apply(
    lambda row: get_nearest_rows(
        edges[edges['highway'] == 'service'],
        Point(row.x, row.y),
        150
    )['length'].sum(),
    axis=1
).astype(float)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Length of nearby service streets within 150 meters"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = "Depends on the subject"
docs[feature_name]["created_on"] = "osmnx.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
with open("documentation/feature_docs.toml", "w") as f:
    toml.dump(docs, f)

In [ ]:
df.to_parquet('data/processed_data/S3-approx-coordinates.parquet')